---
title: "Portugal: matriz de Leontief e choques setoriais"
subtitle: "Fonte principal: OECD Input-Output Tables (TTL, raw)"
lang: pt-PT
jupyter: py313
format:
  html:
    toc: true
    toc-location: left
    number-sections: true
    theme: cosmo
    from: markdown+tex_math_single_backslash
execute:
  echo: false
  warning: false
  message: false
  cache: true
---

## Introdução e objetivo

Este módulo continua a análise de matrizes de entradas-saídas iniciada no módulo 2, agora com foco no sistema de Leontief em detalhe setorial **raw** da OECD para Portugal.

Para estudantes de licenciatura em Introdução à Macroeconomia, a ideia central é simples: um choque num setor não afeta apenas esse setor. Como as empresas compram e vendem entre si, há efeitos em cadeia (diretos e indiretos) ao longo da rede produtiva. A matriz de Leontief é uma forma transparente de medir esses encadeamentos.

Neste contexto, vamos trabalhar com contas em valor (milhões de USD), não em quantidades físicas. Assim, cada resultado deve ser lido como um exercício de contabilidade económica e propagação setorial, útil para organizar o raciocínio macro, antes de modelos mais avançados com preços relativos, substituição e dinâmica intertemporal.

Uma nota importante sobre as linhas especiais da base OECD `ttl`:

- `TXS_INT_FNL` representa **impostos líquidos de subsídios sobre produtos** alocados aos usos intermédios e finais (conceito `TLS` no ReadMe da ICIO);
- `IMP_OTHER` representa **outras importações** (um ajustamento de importações fora das linhas setoriais `TTL_*`).

No módulo, estas duas linhas são tratadas como “setores fictícios” que vendem a todos os setores, mas não compram inputs intermédios, para manter a leitura económica do sistema de Leontief com dados raw.

Referência principal da fonte e documentação: [OECD Inter-Country Input-Output (ICIO)](https://www.oecd.org/en/data/datasets/inter-country-input-output-tables.html).

Os objetivos são:

- construir o sistema
  $$
  x = Ax + y,
  $$
  onde as colunas de $A$ são compradores e as linhas são vendedores;
- incluir `TXS_INT_FNL` e `IMP_OTHER` como setores fictícios (`TXS` e `IMP_OTHER`) que vendem a todos, mas não compram a ninguém;
- interpretar a matriz técnica $A$ e a inversa de Leontief $(I-A)^{-1}$;
- ligar os sistemas de quantidades e preços;
- estudar dois exercícios de política de procura final:
  - choque de `-1%` em `H51` (Air transport),
  - choque de `+1%` em `I` (Accommodation and food service activities; aqui `I` é código setorial, não investimento).

In [ ]:
#| label: setup
#| include: false
import io
import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests

OECD_ZIP_URL = "https://stats.oecd.org/wbos/fileview2.aspx?IDFile=67d903bc-edff-463c-aeeb-52d711731221"

FD_COLS = [
    "HFCE",
    "NPISH",
    "GGFC",
    "GFCF",
    "INVNT",
    "DPABR",
    "CONS_NONRES",
    "EXPO",
    "IMPO",
]

REQUIRED_ROWS = ["OUTPUT", "TXS_INT_FNL", "IMP_OTHER", "TTL_H51", "TTL_I"]
TARGET_SHOCKS = {"H51": -0.01, "I": 0.01}

SECTOR_LABELS_PT = {
    "A01": "Agricultura e caça",
    "A02": "Silvicultura",
    "A03": "Pesca e aquicultura",
    "B07": "Extração de minérios metálicos",
    "B08": "Outras indústrias extrativas",
    "B09": "Serviços de apoio à extração",
    "C10T12": "Alimentares, bebidas e tabaco",
    "C13T15": "Têxteis, vestuário e couro",
    "C16": "Madeira e cortiça",
    "C17_18": "Papel e impressão",
    "C19": "Refinação de petróleo",
    "C20": "Químicos",
    "C21": "Farmacêuticos",
    "C22": "Borracha e plásticos",
    "C23": "Minerais não metálicos",
    "C24A": "Metais ferrosos",
    "C24B": "Metais não ferrosos",
    "C25": "Produtos metálicos",
    "C26": "Eletrónica e informática",
    "C27": "Equipamento elétrico",
    "C28": "Máquinas e equipamentos",
    "C29": "Veículos automóveis",
    "C301": "Construção naval",
    "C302T309": "Outro equipamento de transporte",
    "C31T33": "Mobiliário e outras indústrias",
    "D": "Eletricidade, gás e vapor",
    "E": "Água, saneamento e resíduos",
    "F": "Construção",
    "G": "Comércio",
    "H49": "Transporte terrestre",
    "H50": "Transporte marítimo",
    "H51": "Transporte aéreo",
    "H52": "Armazenagem e apoio ao transporte",
    "H53": "Correio e estafetas",
    "I": "Alojamento e restauração",
    "J58T60": "Edição, audiovisual e media",
    "J61": "Telecomunicações",
    "J62_63": "Serviços informáticos e informação",
    "K": "Atividades financeiras e seguros",
    "L": "Imobiliário",
    "M": "Atividades profissionais e científicas",
    "N": "Atividades administrativas e apoio",
    "O": "Administração pública e defesa",
    "P": "Educação",
    "Q": "Saúde e apoio social",
    "R": "Artes, cultura e recreação",
    "S": "Outros serviços",
    "T": "Serviços domésticos das famílias",
    "TXS": "Impostos líquidos sobre produtos",
    "IMP_OTHER": "Outras importações",
}

ARTIFACTS_DIR = Path("03_portugal_iot_leontief_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def sector_label(code: str):
    return SECTOR_LABELS_PT.get(code, code)


def parse_year(name: str):
    match = re.search(r"(\d{4})", name)
    return int(match.group(1)) if match else None


def load_local_prt_ttl():
    candidates = []
    for p in Path(".").glob("PRT*ttl.csv"):
        year = parse_year(p.name)
        if year is None:
            continue
        if p.stat().st_size <= 0:
            continue
        candidates.append((year, p))

    if not candidates:
        return None, None, None

    candidates.sort(key=lambda t: t[0])
    year, path = candidates[-1]
    df = pd.read_csv(path, index_col=0)
    return df, year, path.name


def load_remote_prt_ttl():
    resp = requests.get(OECD_ZIP_URL, timeout=60)
    resp.raise_for_status()

    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    prt_files = [n for n in zf.namelist() if n.startswith("PRT") and n.endswith("ttl.csv")]
    if not prt_files:
        raise ValueError("Não foi possível encontrar ficheiros PRT*ttl.csv no ZIP OECD.")

    years = []
    for n in prt_files:
        y = parse_year(n)
        if y is not None:
            years.append((y, n))

    if not years:
        raise ValueError("Não foi possível extrair o ano dos ficheiros PRT*ttl.csv.")

    years.sort(key=lambda t: t[0])
    year, chosen = years[-1]
    with zf.open(chosen) as f:
        df = pd.read_csv(f, index_col=0)

    return df, year, chosen


def load_oecd_prt_ttl_raw():
    local_df, local_year, local_name = load_local_prt_ttl()
    if local_df is not None:
        return local_df, local_year, local_name, "local"

    remote_df, remote_year, remote_name = load_remote_prt_ttl()
    return remote_df, remote_year, remote_name, "download"


def build_leontief_ext(table: pd.DataFrame):
    if "TOTAL" not in table.columns:
        raise ValueError("A coluna TOTAL é obrigatória.")

    for row in REQUIRED_ROWS:
        if row not in table.index:
            raise ValueError(f"Linha obrigatória em falta: {row}")

    for col in FD_COLS:
        if col not in table.columns:
            raise ValueError(f"Coluna de procura final em falta: {col}")

    sector_cols = [c for c in table.columns if c not in FD_COLS + ["TOTAL"]]

    if "H51" not in sector_cols or "I" not in sector_cols:
        raise ValueError("Os setores H51 e I têm de existir nas colunas setoriais.")

    x_obs = table.loc["OUTPUT", sector_cols].astype(float)

    S_raw = [j for j in sector_cols if x_obs.loc[j] > 0]
    dropped_non_produced = [j for j in sector_cols if x_obs.loc[j] <= 0]

    S = []
    dropped_missing_ttl = []
    for j in S_raw:
        ttl_row = f"TTL_{j}"
        if ttl_row in table.index:
            S.append(j)
        else:
            dropped_missing_ttl.append(j)

    if "H51" not in S or "I" not in S:
        raise ValueError("H51 e I têm de estar presentes em S (setores produzidos com linha TTL_ correspondente).")

    sellers_ext = S + ["TXS", "IMP_OTHER"]

    Z = pd.DataFrame(0.0, index=sellers_ext, columns=S)
    for i in S:
        Z.loc[i, S] = table.loc[f"TTL_{i}", S].astype(float).values

    Z.loc["TXS", S] = table.loc["TXS_INT_FNL", S].astype(float).values
    Z.loc["IMP_OTHER", S] = table.loc["IMP_OTHER", S].astype(float).values

    x_ext = pd.Series(index=sellers_ext, dtype=float)
    x_ext.loc[S] = x_obs.loc[S].values
    x_ext.loc["TXS"] = float(table.loc["TXS_INT_FNL", "TOTAL"])
    x_ext.loc["IMP_OTHER"] = float(table.loc["IMP_OTHER", "TOTAL"])

    A_ext = pd.DataFrame(0.0, index=sellers_ext, columns=sellers_ext)
    for j in S:
        A_ext.loc[sellers_ext, j] = Z.loc[sellers_ext, j] / x_ext.loc[j]

    # Colunas dos setores fictícios (compradores) são zero por construção.
    A_ext.loc[:, "TXS"] = 0.0
    A_ext.loc[:, "IMP_OTHER"] = 0.0

    y_ext = pd.Series(index=sellers_ext, dtype=float)
    for r in S:
        y_ext.loc[r] = float(table.loc[f"TTL_{r}", FD_COLS].astype(float).sum())
    y_ext.loc["TXS"] = float(table.loc["TXS_INT_FNL", FD_COLS].astype(float).sum())
    y_ext.loc["IMP_OTHER"] = float(table.loc["IMP_OTHER", FD_COLS].astype(float).sum())

    I_ext = np.eye(len(sellers_ext))
    M_ext = I_ext - A_ext.values

    inverse_ok = True
    try:
        L_values = np.linalg.inv(M_ext)
    except np.linalg.LinAlgError:
        inverse_ok = False
        L_values = np.linalg.pinv(M_ext)

    L_ext = pd.DataFrame(L_values, index=sellers_ext, columns=sellers_ext)
    x_hat = pd.Series(L_ext.values @ y_ext.values, index=sellers_ext)

    balance_error = x_ext - (A_ext @ x_ext + y_ext)
    max_abs_balance_error = float(balance_error.abs().max())
    max_rel_balance_error = float(
        (balance_error.abs() / x_ext.abs().replace(0, np.nan)).fillna(0).max()
    )

    xhat_error = x_hat - x_ext
    max_abs_xhat_error = float(xhat_error.abs().max())
    max_rel_xhat_error = float(
        (xhat_error.abs() / x_ext.abs().replace(0, np.nan)).fillna(0).max()
    )

    A_prod = A_ext.loc[S, S]
    spectral_radius_A_prod = float(np.max(np.abs(np.linalg.eigvals(A_prod.values))))

    Omega = A_ext.T

    b_denom = float(y_ext.loc[S].sum())
    if np.isclose(b_denom, 0.0):
        raise ValueError("A soma de procura final sobre setores produzidos é zero; não é possível calcular b_i.")
    b_shares = (y_ext.loc[S] / b_denom).rename("b_share")

    dropped_sectors = dropped_non_produced + dropped_missing_ttl

    return {
        "sector_cols": sector_cols,
        "S": S,
        "S_ext": sellers_ext,
        "x_obs": x_obs,
        "Z": Z,
        "x_ext": x_ext,
        "A_ext": A_ext,
        "y_ext": y_ext,
        "L_ext": L_ext,
        "x_hat": x_hat,
        "Omega": Omega,
        "b_shares": b_shares,
        "balance_error": balance_error,
        "max_abs_balance_error": max_abs_balance_error,
        "max_rel_balance_error": max_rel_balance_error,
        "max_abs_xhat_error": max_abs_xhat_error,
        "max_rel_xhat_error": max_rel_xhat_error,
        "inverse_ok": inverse_ok,
        "spectral_radius_A_prod": spectral_radius_A_prod,
        "dropped_sectors": dropped_sectors,
    }


def solve_price_system(table: pd.DataFrame, A_ext: pd.DataFrame, S: list[str]):
    A_ss = A_ext.loc[S, S]
    A_ws = A_ext.loc[["TXS", "IMP_OTHER"], S]

    v_s = (table.loc["VALU", S].astype(float) / table.loc["OUTPUT", S].astype(float)).rename("v_share")
    p_w = np.array([1.0, 1.0])

    rhs = v_s.values + A_ws.values.T @ p_w
    M = np.eye(len(S)) - A_ss.values.T

    price_solve_ok = True
    try:
        p_s = np.linalg.solve(M, rhs)
    except np.linalg.LinAlgError:
        price_solve_ok = False
        p_s = np.linalg.pinv(M) @ rhs

    p_s = pd.Series(p_s, index=S, name="p_hat")
    p_ext = pd.concat([p_s, pd.Series({"TXS": 1.0, "IMP_OTHER": 1.0}, name="p_hat")])

    price_components = pd.DataFrame(
        {
            "v_share": v_s,
            "txs_direct": A_ext.loc["TXS", S].astype(float),
            "imp_other_direct": A_ext.loc["IMP_OTHER", S].astype(float),
            "p_hat": p_s,
        }
    )

    return p_ext, price_components, price_solve_ok


def shock_response(L_ext: pd.DataFrame, x_ext: pd.Series, y_ext: pd.Series, sector: str, pct: float):
    dy = pd.Series(0.0, index=y_ext.index)
    dy.loc[sector] = pct * float(y_ext.loc[sector])

    dx = L_ext @ dy
    rel = (dx / x_ext.replace(0, np.nan)).fillna(0.0)

    out = pd.DataFrame(
        {
            "sector": dx.index,
            "delta_x": dx.values,
            "delta_x_pct": 100 * rel.values,
        }
    )
    out["abs_delta_x"] = out["delta_x"].abs()
    out = out.sort_values("abs_delta_x", ascending=False)
    return out, dy


def save_artifacts(artifacts_dir: Path, payload: dict, price_components: pd.DataFrame, check_report: pd.DataFrame):
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    payload["A_ext"].to_csv(artifacts_dir / "A_ext.csv", index_label="seller")

    payload["x_ext"].rename("x_ext").rename_axis("sector").reset_index().to_csv(
        artifacts_dir / "x_ext.csv", index=False
    )

    payload["y_ext"].rename("y_ext").rename_axis("sector").reset_index().to_csv(
        artifacts_dir / "y_ext.csv", index=False
    )

    payload["L_ext"].to_csv(artifacts_dir / "leontief_inverse_ext.csv", index_label="seller")
    payload["Omega"].to_csv(artifacts_dir / "omega_ext.csv", index_label="buyer")

    payload["b_shares"].rename_axis("sector").reset_index().to_csv(
        artifacts_dir / "b_shares.csv", index=False
    )

    price_components.rename_axis("sector").reset_index().to_csv(
        artifacts_dir / "price_components.csv", index=False
    )

    check_report.to_csv(artifacts_dir / "check_report.csv", index=False)

In [ ]:
#| label: leontief-build
#| include: false
raw_table, latest_year, source_file, source_mode = load_oecd_prt_ttl_raw()
result = build_leontief_ext(raw_table)

A_ext = result["A_ext"]
x_ext = result["x_ext"]
y_ext = result["y_ext"]
L_ext = result["L_ext"]
Omega = result["Omega"]
b_shares = result["b_shares"]
S = result["S"]
S_ext = result["S_ext"]

p_ext, price_components, price_solve_ok = solve_price_system(raw_table, A_ext, S)

report = pd.DataFrame(
    [
        {
            "year": int(latest_year),
            "source_file": source_file,
            "source_mode": source_mode,
            "n_sector_cols": len(result["sector_cols"]),
            "n_produced": len(S),
            "dropped_sectors": "|".join(result["dropped_sectors"]),
            "max_abs_balance_error": result["max_abs_balance_error"],
            "max_rel_balance_error": result["max_rel_balance_error"],
            "max_abs_xhat_error": result["max_abs_xhat_error"],
            "max_rel_xhat_error": result["max_rel_xhat_error"],
            "inverse_ok": bool(result["inverse_ok"]),
            "price_solve_ok": bool(price_solve_ok),
            "spectral_radius_A_prod": result["spectral_radius_A_prod"],
        }
    ]
)

save_artifacts(ARTIFACTS_DIR, result, price_components, report)

shock_h51, dy_h51 = shock_response(L_ext, x_ext, y_ext, "H51", TARGET_SHOCKS["H51"])
shock_i, dy_i = shock_response(L_ext, x_ext, y_ext, "I", TARGET_SHOCKS["I"])

multiplier_summary = pd.DataFrame(
    {
        "setor_codigo": ["H51", "I"],
        "setor": [sector_label("H51"), sector_label("I")],
        "choque_pct": [100 * TARGET_SHOCKS["H51"], 100 * TARGET_SHOCKS["I"]],
        "delta_y": [dy_h51.loc["H51"], dy_i.loc["I"]],
        "multiplicador_output_total": [
            float((L_ext @ dy_h51).loc[S].sum() / dy_h51.loc["H51"]),
            float((L_ext @ dy_i).loc[S].sum() / dy_i.loc["I"]),
        ],
    }
)

v_share = price_components["v_share"].astype(float).reindex(S)

scenario_specs = [
    ("H51", TARGET_SHOCKS["H51"], dy_h51, shock_h51),
    ("I", TARGET_SHOCKS["I"], dy_i, shock_i),
]

scenario_rows = []
for code, shock_pct, dy_vec, shock_tbl in scenario_specs:
    dx_vec = L_ext @ dy_vec
    others = shock_tbl[shock_tbl["sector"] != code]

    delta_y = float(dy_vec.loc[code])
    delta_x_total = float(dx_vec.loc[S].sum())
    delta_va = float((v_share * dx_vec.loc[S]).sum())
    delta_txs = float(dx_vec.loc["TXS"])
    delta_imp_other = float(dx_vec.loc["IMP_OTHER"])

    output_multiplier = np.nan if np.isclose(delta_y, 0.0) else delta_x_total / delta_y
    va_multiplier = np.nan if np.isclose(delta_y, 0.0) else delta_va / delta_y
    txs_per_delta_y = np.nan if np.isclose(delta_y, 0.0) else delta_txs / delta_y
    imp_other_per_delta_y = np.nan if np.isclose(delta_y, 0.0) else delta_imp_other / delta_y
    va_share_in_delta_x = np.nan if np.isclose(delta_x_total, 0.0) else delta_va / delta_x_total

    scenario_rows.append(
        {
            "setor_codigo": code,
            "setor": sector_label(code),
            "choque_pct": 100 * shock_pct,
            "delta_y": delta_y,
            "delta_x_total": delta_x_total,
            "delta_va": delta_va,
            "delta_txs": delta_txs,
            "delta_imp_other": delta_imp_other,
            "output_multiplier": output_multiplier,
            "va_multiplier": va_multiplier,
            "txs_per_delta_y": txs_per_delta_y,
            "imp_other_per_delta_y": imp_other_per_delta_y,
            "va_share_in_delta_x": va_share_in_delta_x,
            "n_remaining_sectors": int(len(others)),
            "n_pos_remaining": int((others["delta_x"] > 0).sum()),
            "n_neg_remaining": int((others["delta_x"] < 0).sum()),
            "n_zero_remaining": int((others["delta_x"] == 0).sum()),
        }
    )

scenario_summary = pd.DataFrame(scenario_rows)
scenario_lookup = scenario_summary.set_index("setor_codigo")

L_prod = L_ext.loc[S, S].astype(float)
backward_linkages = L_prod.sum(axis=0).sort_values(ascending=False).rename("backward_linkage")
forward_linkages = L_prod.sum(axis=1).sort_values(ascending=False).rename("forward_linkage")

backward_top10 = backward_linkages.head(10).rename_axis("sector").reset_index()
backward_top10["Setor"] = backward_top10["sector"].map(lambda c: f"{sector_label(c)} ({c})")
backward_top10 = backward_top10[["Setor", "backward_linkage"]]

forward_top10 = forward_linkages.head(10).rename_axis("sector").reset_index()
forward_top10["Setor"] = forward_top10["sector"].map(lambda c: f"{sector_label(c)} ({c})")
forward_top10 = forward_top10[["Setor", "forward_linkage"]]

SANDBOX_SHOCK_GRID = [-1.0, 1.0]
sandbox_top_rows = []
sandbox_summary_rows = []

for shock_sector in S:
    for shock_pct in SANDBOX_SHOCK_GRID:
        dy_tmp = pd.Series(0.0, index=y_ext.index)
        dy_tmp.loc[shock_sector] = (shock_pct / 100.0) * float(y_ext.loc[shock_sector])

        dx_tmp = L_ext @ dy_tmp
        rel_tmp = (dx_tmp / x_ext.replace(0, np.nan)).fillna(0.0)

        tmp = pd.DataFrame(
            {
                "sector": S,
                "delta_x": dx_tmp.loc[S].values,
                "delta_x_pct": 100 * rel_tmp.loc[S].values,
            }
        )
        tmp["abs_delta_x"] = tmp["delta_x"].abs()
        tmp = tmp.sort_values("abs_delta_x", ascending=False).head(15)
        tmp["shock_sector"] = shock_sector
        tmp["shock_pct"] = shock_pct
        sandbox_top_rows.append(tmp[["shock_sector", "shock_pct", "sector", "delta_x", "delta_x_pct", "abs_delta_x"]])

        delta_y_tmp = float(dy_tmp.loc[shock_sector])
        delta_x_total_tmp = float(dx_tmp.loc[S].sum())
        delta_va_tmp = float((v_share * dx_tmp.loc[S]).sum())
        delta_txs_tmp = float(dx_tmp.loc["TXS"])
        delta_imp_other_tmp = float(dx_tmp.loc["IMP_OTHER"])

        output_mult_tmp = np.nan if np.isclose(delta_y_tmp, 0.0) else delta_x_total_tmp / delta_y_tmp
        va_mult_tmp = np.nan if np.isclose(delta_y_tmp, 0.0) else delta_va_tmp / delta_y_tmp
        tax_wedge_tmp = np.nan if np.isclose(delta_y_tmp, 0.0) else delta_txs_tmp / delta_y_tmp
        imp_leak_tmp = np.nan if np.isclose(delta_y_tmp, 0.0) else delta_imp_other_tmp / delta_y_tmp

        sandbox_summary_rows.append(
            {
                "shock_sector": shock_sector,
                "shock_pct": shock_pct,
                "delta_y": delta_y_tmp,
                "delta_x_total": delta_x_total_tmp,
                "delta_va": delta_va_tmp,
                "delta_txs": delta_txs_tmp,
                "delta_imp_other": delta_imp_other_tmp,
                "output_multiplier": output_mult_tmp,
                "va_multiplier": va_mult_tmp,
                "tax_wedge_ratio": tax_wedge_tmp,
                "import_leakage_ratio": imp_leak_tmp,
            }
        )

sandbox_top_impacts = pd.concat(sandbox_top_rows, ignore_index=True)
sandbox_summary = pd.DataFrame(sandbox_summary_rows)
sandbox_summary["scenario_label"] = sandbox_summary.apply(
    lambda r: f"{sector_label(r['shock_sector'])} ({r['shock_sector']}) | choque {r['shock_pct']:+.1f}%",
    axis=1,
)

A_ss = A_ext.loc[S, S].astype(float)
A_ws = A_ext.loc[["TXS", "IMP_OTHER"], S].astype(float)
M_price = np.eye(len(S)) - A_ss.values.T

price_inverse_ok = True
try:
    price_propagator_values = np.linalg.inv(M_price)
except np.linalg.LinAlgError:
    price_inverse_ok = False
    price_propagator_values = np.linalg.pinv(M_price)

price_propagator = pd.DataFrame(price_propagator_values, index=S, columns=S)

delta_p_w = pd.Series({"TXS": 0.0, "IMP_OTHER": 0.10})
rhs_import_case = A_ws.values.T @ delta_p_w.loc[["TXS", "IMP_OTHER"]].values
delta_p_s = pd.Series(price_propagator.values @ rhs_import_case, index=S, name="delta_p_hat")

import_cost_case = pd.DataFrame({"sector": S, "delta_p_hat": delta_p_s.values})
import_cost_case["p_hat_base"] = price_components.loc[S, "p_hat"].values
import_cost_case["delta_p_hat_pct_base"] = (
    100 * import_cost_case["delta_p_hat"] / import_cost_case["p_hat_base"].replace(0, np.nan)
)
import_cost_case["abs_delta_p_hat"] = import_cost_case["delta_p_hat"].abs()
import_cost_case_top15 = import_cost_case.sort_values("abs_delta_p_hat", ascending=False).head(15).copy()
import_cost_case_top15["Setor"] = import_cost_case_top15["sector"].map(lambda c: f"{sector_label(c)} ({c})")

In [ ]:
#| label: txt-data-metadata
#| echo: false
#| output: asis
print(
    f"**Dados:** OECD Input-Output Tables (TTL, raw), Portugal, ano = {int(latest_year)}, "
    f"preços correntes; unidades: milhões de USD (fonte carregada: `{source_file}`)."
)

## Construção de $x$, $A$ e $y$ com setores fictícios

A construção segue, exatamente, os passos:

1. setores produzidos com `OUTPUT > 0`;
2. matriz de vendas intermédias $Z$ com linhas vendedores e colunas compradores;
3. extensão com `TXS` e `IMP_OTHER` como vendedores fictícios;
4. matriz técnica estendida $A_{ext}$ com colunas fictícias de compradores iguais a zero;
5. procura final líquida $y_{ext}$ pela soma das colunas finais (incluindo `IMPO`, negativa).

Formalmente:

$$
x_{ext} = A_{ext}x_{ext} + y_{ext}.
$$

O tratamento de `TXS` e `IMP_OTHER` é importante para manter a interpretação económica correta:

- `TXS` (impostos líquidos sobre produtos) e `IMP_OTHER` (outras importações) entram como **linhas vendedoras**. Isto significa que contam como custos por unidade de produção dos setores compradores.
- Ao mesmo tempo, são definidos como setores que **não compram inputs intermédios** no sistema (`A_ext[:, "TXS"] = 0` e `A_ext[:, "IMP_OTHER"] = 0`).
- Em termos práticos, funcionam como “cunhas” (wedges) contabilísticas: afetam custos e preços de produção, mas não criam rondas adicionais de procura intermédia como um setor produtivo convencional.

::: {.callout-note}
## Nota sobre importações no modelo
`IMPO` entra na procura final como termo negativo (procura final líquida).  
`IMP_OTHER` entra como setor fictício vendedor: funciona como cunha de custo importado por unidade de produção e não gera novas rondas intermédias, porque não compra inputs no sistema.
:::

In [ ]:
#| label: tbl-system-dims
#| tbl-cap: "Dimensões principais do sistema estendido."
tbl_dims = pd.DataFrame(
    {
        "Objeto": ["Setores originais (colunas)", "Setores produzidos (S)", "Setores estendidos (S_ext)", "Dimensão de A_ext"],
        "Valor": [
            len(result["sector_cols"]),
            len(S),
            len(S_ext),
            f"{A_ext.shape[0]} x {A_ext.shape[1]}",
        ],
    }
)

(
    tbl_dims.style.hide(axis="index")
    .set_properties(subset=["Objeto"], **{"text-align": "left"})
    .set_properties(subset=["Valor"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {"selector": "th.col_heading.col1", "props": [("text-align", "right")]},
        ],
        overwrite=False,
    )
)

A Tabela 2 é um teste de coerência interna do sistema. A identidade $x = Ax + y$ deve ser satisfeita (até erro numérico de arredondamento) se a construção de $A_{ext}$, $x_{ext}$ e $y_{ext}$ estiver correta. Mostramos setores económicos (`H51` e `I`) e os dois setores fictícios (`TXS` e `IMP_OTHER`) para confirmar que a lógica é consistente em todos os blocos do modelo.

In [ ]:
#| label: tbl-equation-check
#| tbl-cap: "Verificação da identidade x = Ax + y (setores selecionados)."
check_tbl = pd.DataFrame(
    {
        "setor": x_ext.index,
        "x_ext": x_ext,
        "A_ext_x_plus_y": (A_ext @ x_ext + y_ext),
        "erro": result["balance_error"],
    }
)
check_tbl["setor_descricao"] = check_tbl["setor"].map(sector_label)
check_tbl["setor"] = check_tbl["setor_descricao"]

focus_sectors = ["H51", "I", "TXS", "IMP_OTHER"]
focus_labels = [sector_label(s) for s in focus_sectors]
focus_labels = [s for s in focus_labels if s in set(check_tbl["setor"])]

(
    check_tbl[check_tbl["setor"].isin(focus_labels)]
    .loc[:, ["setor", "x_ext", "A_ext_x_plus_y", "erro"]]
    .style.hide(axis="index")
    .format({"x_ext": "{:,.2f}", "A_ext_x_plus_y": "{:,.2f}", "erro": "{:.6f}"})
    .set_properties(subset=["setor"], **{"text-align": "left"})
    .set_properties(subset=["x_ext", "A_ext_x_plus_y", "erro"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {
                "selector": "th.col_heading.col1, th.col_heading.col2, th.col_heading.col3",
                "props": [("text-align", "right")],
            },
        ],
        overwrite=False,
    )
    .set_properties(**{"font-size": "0.9em"})
)

## Interpretação da matriz técnica $A$

Cada entrada da matriz técnica tem interpretação direta:

$$
a_{ij} = \frac{z_{ij}}{x_j},
$$

onde $z_{ij}$ é a venda intermédia do setor vendedor $i$ ao setor comprador $j$ e $x_j$ é o output bruto de $j$.

Assim, **cada elemento** $a_{ij}$ mede quantas unidades monetárias de input de $i$ são necessárias, diretamente, para produzir 1 unidade monetária de output de $j$.

Leituras úteis:

- Entrada diagonal `a_{jj}`: uso interno direto do próprio setor $j$.
- Entrada fora da diagonal `a_{ij}` com `i ≠ j`: dependência direta de $j$ em relação ao setor fornecedor $i$.
- Coluna $j$ de $A$: vetor completo da estrutura de custos intermédios diretos do setor $j$.

In [ ]:
#| label: tbl-a-columns-targets
#| tbl-cap: "Principais coeficientes técnicos diretos (a_ij) para os setores transporte aéreo e alojamento/restauração."
rows = []
for buyer in ["H51", "I"]:
    coeff = A_ext.loc[:, buyer].sort_values(ascending=False)
    coeff = coeff[coeff > 0].head(12)
    tmp = coeff.rename("a_ij").reset_index()
    tmp.columns = ["setor_vendedor_codigo", "a_ij"]
    tmp["Setor comprador"] = sector_label(buyer)
    tmp["Setor vendedor"] = tmp["setor_vendedor_codigo"].map(sector_label)
    rows.append(tmp[["Setor comprador", "Setor vendedor", "a_ij"]])

tbl_a = pd.concat(rows, ignore_index=True)

(
    tbl_a.style.hide(axis="index")
    .format({"a_ij": "{:.4f}"})
    .set_properties(subset=["Setor comprador", "Setor vendedor"], **{"text-align": "left"})
    .set_properties(subset=["a_ij"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0, th.col_heading.col1", "props": [("text-align", "left")]},
            {"selector": "th.col_heading.col2", "props": [("text-align", "right")]},
        ],
        overwrite=False,
    )
)

## Interpretação da inversa de Leontief $(I-A)^{-1}$

Com

$$
x = (I-A)^{-1}y \equiv Ly,
$$

cada elemento

$$
l_{ij} = \left[(I-A)^{-1}\right]_{ij}
$$

tem interpretação económica precisa: é a variação total do output do setor $i$ quando a procura final do setor $j$ aumenta em 1 unidade monetária, mantendo fixos os coeficientes técnicos.

Uma forma útil de ver esta matriz é pela expansão em série:

$$
(I-A)^{-1} = I + A + A^2 + A^3 + \cdots
$$

Se quisermos olhar apenas para os encadeamentos para além do efeito “imediato” no próprio setor, então:

$$
(I-A)^{-1} - I = A + A^2 + A^3 + \cdots
$$

Isto inclui:

- efeito de 1.ª ronda (produção intermédia direta, $A\Delta y$),
- efeitos indiretos (rondas sucessivas, $A^2\Delta y + A^3\Delta y + \cdots$).

Leituras úteis:

- Entrada diagonal `l_{jj}`: efeito total no próprio setor após todas as rondas.
- Entrada fora da diagonal `l_{ij}`: transmissão intersetorial de um choque em $j$ para o setor $i$.
- Coluna $j$ de $L$: distribuição setorial completa do multiplicador de um choque em $y_j$.

In [ ]:
#| label: tbl-multipliers
#| tbl-cap: "Multiplicadores agregados (output total produzido) para choques em H51 e I."
multiplier_display = multiplier_summary.drop(columns=["setor_codigo"]).rename(columns={"setor": "Setor do choque"})

(
    multiplier_display.style.hide(axis="index")
    .format(
        {
            "choque_pct": "{:.2f}",
            "delta_y": "{:,.2f}",
            "multiplicador_output_total": "{:.4f}",
        }
    )
    .set_properties(subset=["Setor do choque"], **{"text-align": "left"})
    .set_properties(subset=["choque_pct", "delta_y", "multiplicador_output_total"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {
                "selector": "th.col_heading.col1, th.col_heading.col2, th.col_heading.col3",
                "props": [("text-align", "right")],
            },
        ],
        overwrite=False,
    )
)

Os multiplicadores da Tabela 4 mostram que o efeito agregado no output ultrapassa o choque inicial de procura final em ambos os casos. Em termos económicos, isto reflete rondas sucessivas de produção intermédia: um aumento (ou queda) de procura num setor altera as compras a fornecedores, que por sua vez ajustam a sua própria procura de inputs.

In [ ]:
#| label: txt-multipliers-interpretation
#| echo: false
#| output: asis
h51_m = float(multiplier_summary.loc[multiplier_summary["setor_codigo"] == "H51", "multiplicador_output_total"].iloc[0])
i_m = float(multiplier_summary.loc[multiplier_summary["setor_codigo"] == "I", "multiplicador_output_total"].iloc[0])

h51_dy_abs = abs(float(multiplier_summary.loc[multiplier_summary["setor_codigo"] == "H51", "delta_y"].iloc[0]))
i_dy_abs = abs(float(multiplier_summary.loc[multiplier_summary["setor_codigo"] == "I", "delta_y"].iloc[0]))

txt = f"""
Leitura de alguns números:

- Um choque de 1 unidade monetária na procura final de **{sector_label('H51')}** gera, em média, **{h51_m:.3f}** unidades de variação no output total.
- Um choque de 1 unidade monetária na procura final de **{sector_label('I')}** gera, em média, **{i_m:.3f}** unidades de variação no output total.
- Na nossa calibração, os choques aplicados têm magnitude de **{h51_dy_abs:,.1f}** e **{i_dy_abs:,.1f} milhões USD**, respetivamente, e por isso os efeitos agregados diferem também pela dimensão inicial de Δy.
"""
print(txt)

In [ ]:
#| label: tbl-leontief-columns-targets
#| tbl-cap: "Principais coeficientes da inversa de Leontief (l_ij) para choques unitários em transporte aéreo e alojamento/restauração."
rows = []
for buyer in ["H51", "I"]:
    lcol = L_ext.loc[S, buyer].sort_values(ascending=False).head(12)
    tmp = lcol.rename("l_ij").reset_index()
    tmp.columns = ["setor_afetado_codigo", "l_ij"]
    tmp["Setor do choque"] = sector_label(buyer)
    tmp["Setor afetado"] = tmp["setor_afetado_codigo"].map(sector_label)
    rows.append(tmp[["Setor do choque", "Setor afetado", "l_ij"]])

tbl_l = pd.concat(rows, ignore_index=True)

(
    tbl_l.style.hide(axis="index")
    .format({"l_ij": "{:.4f}"})
    .set_properties(subset=["Setor do choque", "Setor afetado"], **{"text-align": "left"})
    .set_properties(subset=["l_ij"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0, th.col_heading.col1", "props": [("text-align", "left")]},
            {"selector": "th.col_heading.col2", "props": [("text-align", "right")]},
        ],
        overwrite=False,
    )
)

A Tabela 5 decompõe os efeitos totais por setor afetado. Quanto maior for $l_{ij}$, maior é a sensibilidade do setor $i$ a um choque de procura final no setor $j$. Isto ajuda a identificar ligações produtivas particularmente fortes.

In [ ]:
#| label: txt-leontief-interpretation
#| echo: false
#| output: asis
l_h51 = L_ext.loc[S, "H51"].sort_values(ascending=False)
l_i = L_ext.loc[S, "I"].sort_values(ascending=False)

spill_h51 = l_h51.drop(labels=["H51"], errors="ignore").head(1)
spill_i = l_i.drop(labels=["I"], errors="ignore").head(1)

top_h51_sector = spill_h51.index[0]
top_h51_value = float(spill_h51.iloc[0])
top_i_sector = spill_i.index[0]
top_i_value = float(spill_i.iloc[0])

txt = f"""
Exemplos de leitura económica:

- Entre os efeitos de propagação de um choque em **{sector_label('H51')}**, o maior coeficiente fora da diagonal é para **{sector_label(top_h51_sector)}**, com **l_ij = {top_h51_value:.4f}**.
- Entre os efeitos de propagação de um choque em **{sector_label('I')}**, o maior coeficiente fora da diagonal é para **{sector_label(top_i_sector)}**, com **l_ij = {top_i_value:.4f}**.

Isto sugere que estes setores estão relativamente mais expostos aos encadeamentos indiretos associados aos dois choques analisados.
"""
print(txt)

## Quantidades vs. preços

No bloco de quantidades, resolvemos:

$$
x = Ax + y,
$$

que determina o output necessário em cada setor para acomodar a procura final $y$.

Para preços, a lógica é dual: em vez de “quantidades produzidas”, olhamos para “custos unitários”. Para um setor produzido $i \in S$, a condição de custo unitário é:

$$
p_i = \sum_{j \in S} a_{ji}p_j + a_{\text{TXS},i}p_{\text{TXS}} + a_{\text{IMP\_OTHER},i}p_{\text{IMP\_OTHER}} + v_i,
$$

onde:

- $v_i \equiv VA_i/x_i$ é o valor acrescentado por unidade de output;
- $a_{\text{TXS},i}$ e $a_{\text{IMP\_OTHER},i}$ são as cargas diretas das duas cunhas no setor $i$;
- $p_{\text{TXS}}$ e $p_{\text{IMP\_OTHER}}$ são preços exógenos dessas cunhas.

Empilhando todos os setores produzidos:

$$
p_S = A_{SS}^{\top}p_S + A_{WS}^{\top}p_W + v_S,
$$

com $p_W \equiv (p_{\text{TXS}}, p_{\text{IMP\_OTHER}})^\top$ e $v_S \equiv (VA_i/x_i)_{i \in S}$.

Rearranjando:

$$
(I - A_{SS}^{\top})p_S = v_S + A_{WS}^{\top}p_W,
$$

e, quando a inversa existe,

$$
p_S = (I - A_{SS}^{\top})^{-1}(v_S + A_{WS}^{\top}p_W).
$$

Na Tabela 6 mostramos **$\hat p_i$**, definido como a solução acima sob a normalização:

$$
p_{\text{TXS}} = p_{\text{IMP\_OTHER}} = 1
\quad \Longrightarrow \quad
\hat p_S \equiv (I - A_{SS}^{\top})^{-1}(v_S + A_{WS}^{\top}\mathbf{1}),
$$

isto é, para cada setor $i$:

$$
\hat p_i = \left[(I - A_{SS}^{\top})^{-1}(v_S + A_{WS}^{\top}\mathbf{1})\right]_i.
$$

Assim, `p_hat` deve ser lido como um índice de custo unitário implícito no modelo, não como nível de preço observado diretamente nos dados.
Como os coeficientes técnicos são fixos (sem substituição) e trabalhamos em valores monetários, `p_hat` deve ser interpretado como índice de propagação de custos na rede, e não como um modelo completo de preços de equilíbrio.

In [ ]:
#| label: tbl-price-components-targets
#| tbl-cap: "Componentes diretas de custo e preço implícito (normalizado) para H51 e I."
price_targets = price_components.loc[["H51", "I"]].copy().reset_index().rename(columns={"index": "codigo_setor"})
price_targets["Setor"] = price_targets["codigo_setor"].apply(lambda c: f"{sector_label(c)} ({c})")
price_targets = price_targets[["Setor", "v_share", "txs_direct", "imp_other_direct", "p_hat"]]
(
    price_targets.style.hide(axis="index")
    .format({"v_share": "{:.4f}", "txs_direct": "{:.4f}", "imp_other_direct": "{:.4f}", "p_hat": "{:.4f}"})
    .set_properties(subset=["Setor"], **{"text-align": "left"})
    .set_properties(subset=["v_share", "txs_direct", "imp_other_direct", "p_hat"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {"selector": "th.col_heading.col1, th.col_heading.col2, th.col_heading.col3, th.col_heading.col4", "props": [("text-align", "right")]},
        ],
        overwrite=False,
    )
    .set_properties(**{"font-size": "0.9em"})
)

A Tabela 6 permite comparar a composição direta de custos entre os dois setores de interesse. Em termos pedagógicos:

- `v_share` indica a parcela de valor acrescentado por unidade de output;
- `txs_direct` e `imp_other_direct` mostram o peso direto das duas “cunhas” (impostos líquidos sobre produtos e outras importações);
- `p_hat` resume o efeito total após propagação de custos na rede.

In [ ]:
#| label: txt-price-table-interpretation
#| echo: false
#| output: asis
ph51 = price_components.loc["H51"]
pii = price_components.loc["I"]

txt = f"""
Leitura de alguns números da Tabela 6:

- Em **{sector_label('H51')} (H51)**, a componente direta de valor acrescentado é **{ph51['v_share']:.3f}**, com contribuições diretas de **TXS = {ph51['txs_direct']:.3f}** e **IMP_OTHER = {ph51['imp_other_direct']:.3f}**.
- Em **{sector_label('I')} (I)**, a componente direta de valor acrescentado é **{pii['v_share']:.3f}**, com contribuições diretas de **TXS = {pii['txs_direct']:.3f}** e **IMP_OTHER = {pii['imp_other_direct']:.3f}**.
- O preço implícito normalizado é **{ph51['p_hat']:.3f}** em transporte aéreo e **{pii['p_hat']:.3f}** em alojamento/restauração.
- Interpretação económica de `p_hat`: este indicador resume o custo total unitário após propagação na rede input-output. Um **p_hat** mais elevado sugere maior intensidade de custos (diretos e indiretos) e maior sensibilidade do setor à transmissão de choques de custos vindos de fornecedores e cunhas fiscais/importadas.
"""
print(txt)

In [ ]:
#| label: tbl-price-top-bottom
#| tbl-cap: "Preço implícito normalizado: 5 maiores e 5 menores valores por setor."
price_rank = (
    price_components["p_hat"]
    .sort_values(ascending=False)
    .rename("p_hat")
    .reset_index()
    .rename(columns={"index": "sector"})
)
price_rank["Setor"] = price_rank["sector"].apply(lambda c: f"{sector_label(c)} ({c})")

top5 = price_rank.head(5).copy()
top5["Grupo"] = "Top 5"

bottom5 = price_rank.tail(5).copy().sort_values("p_hat", ascending=True)
bottom5["Grupo"] = "Bottom 5"

price_top_bottom = pd.concat([top5, bottom5], ignore_index=True)

(
    price_top_bottom[["Grupo", "Setor", "p_hat"]]
    .style.hide(axis="index")
    .format({"p_hat": "{:.4f}"})
    .set_properties(subset=["Grupo", "Setor"], **{"text-align": "left"})
    .set_properties(subset=["p_hat"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0, th.col_heading.col1", "props": [("text-align", "left")]},
            {"selector": "th.col_heading.col2", "props": [("text-align", "right")]},
        ],
        overwrite=False,
    )
)

In [ ]:
#| label: txt-price-figure-interpretation
#| echo: false
#| output: asis
top1 = top5.iloc[0]
top2 = top5.iloc[1]
bot1 = bottom5.iloc[0]

txt = f"""
Leitura económica da tabela de extremos de `p_hat`:

- O maior valor de `p_hat` é de **{top1['Setor']}**, com **{top1['p_hat']:.3f}**.
- O segundo maior valor é de **{top2['Setor']}**, com **{top2['p_hat']:.3f}**.
- O menor valor é de **{bot1['Setor']}**, com **{bot1['p_hat']:.3f}**.
- A amplitude entre o maior e o menor `p_hat` na amostra é **{(top1['p_hat'] - bot1['p_hat']):.3f}**, o que evidencia heterogeneidade relevante na intensidade de custos setoriais.
"""
print(txt)

## Exercícios: choques em transporte aéreo e alojamento/restauração

Nesta secção aplicamos dois choques simples de procura final para ilustrar como a rede input-output propaga efeitos entre setores.

Para cada cenário, definimos um vetor de choque $\Delta y$ (com apenas um setor chocado) e calculamos:

$$
\Delta x = L_{ext}\Delta y = (I-A_{ext})^{-1}\Delta y
= \Delta y + (A_{ext} + A_{ext}^2 + A_{ext}^3 + \cdots)\Delta y.
$$

Assim, os resultados apresentados nas figuras são **efeitos totais** sobre o output setorial ($\Delta x$), isto é, incluem:

- efeito inicial (o próprio choque em procura final, $\Delta y$);
- efeito de 1.ª ronda e efeitos indiretos subsequentes (propagação pela rede intermédia, $\Delta x - \Delta y$).

In [ ]:
#| label: tbl-shock-inputs
#| tbl-cap: "Montantes dos choques e grandezas de referência nos dois setores analisados."
shock_inputs = pd.DataFrame(
    {
        "Setor": [f"{sector_label('H51')} (H51)", f"{sector_label('I')} (I)"],
        "Output bruto x (10^6USD)": [float(x_ext.loc["H51"]), float(x_ext.loc["I"])],
        "Procura final y (10^6USD)": [float(y_ext.loc["H51"]), float(y_ext.loc["I"])],
        "Valor acrescentado VA (10^6USD)": [
            float(raw_table.loc["VALU", "H51"]),
            float(raw_table.loc["VALU", "I"]),
        ],
        "Choque (%)": [100 * TARGET_SHOCKS["H51"], 100 * TARGET_SHOCKS["I"]],
        "Choque em procura final Δy (10^6USD)": [float(dy_h51.loc["H51"]), float(dy_i.loc["I"])],
    }
)

(
    shock_inputs.style.hide(axis="index")
    .format(
        {
            "Output bruto x (10^6USD)": "{:,.2f}",
            "Procura final y (10^6USD)": "{:,.2f}",
            "Valor acrescentado VA (10^6USD)": "{:,.2f}",
            "Choque (%)": "{:.2f}",
            "Choque em procura final Δy (10^6USD)": "{:,.2f}",
        }
    )
    .set_properties(subset=["Setor"], **{"text-align": "left"})
    .set_properties(
        subset=[
            "Output bruto x (10^6USD)",
            "Procura final y (10^6USD)",
            "Valor acrescentado VA (10^6USD)",
            "Choque (%)",
            "Choque em procura final Δy (10^6USD)",
        ],
        **{"text-align": "right"},
    )
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {
                "selector": "th.col_heading.col1, th.col_heading.col2, th.col_heading.col3, th.col_heading.col4, th.col_heading.col5",
                "props": [("text-align", "right")],
            },
        ],
        overwrite=False,
    )
)

In [ ]:
#| label: txt-shock-inputs-interpretation
#| echo: false
#| output: asis
h51_output = float(x_ext.loc["H51"])
i_output = float(x_ext.loc["I"])
h51_y = float(y_ext.loc["H51"])
i_y = float(y_ext.loc["I"])
h51_va = float(raw_table.loc["VALU", "H51"])
i_va = float(raw_table.loc["VALU", "I"])
h51_dy = float(dy_h51.loc["H51"])
i_dy = float(dy_i.loc["I"])

txt = f"""
Leitura económica da tabela acima:

- Em **{sector_label('H51')} (H51)**, o output bruto é **{h51_output:,.1f} milhões USD**, a procura final líquida é **{h51_y:,.1f} milhões USD** e o valor acrescentado é **{h51_va:,.1f} milhões USD**; o choque aplicado é **Δy = {h51_dy:,.1f} milhões USD**.
- Em **{sector_label('I')} (I)**, o output bruto é **{i_output:,.1f} milhões USD**, a procura final líquida é **{i_y:,.1f} milhões USD** e o valor acrescentado é **{i_va:,.1f} milhões USD**; o choque aplicado é **Δy = {i_dy:,.1f} milhões USD**.
- Como os choques têm montantes absolutos diferentes, a comparação dos impactos nas figuras deve considerar não só os multiplicadores, mas também a escala inicial de cada $\Delta y$.
"""
print(txt)

### Cenário 1: choque de -1% na procura final de transporte aéreo

In [ ]:
#| label: fig-shock-h51
#| fig-cap: "Cenário 1 (transporte aéreo): impactos no output (top 15 setores)."
plot_h51 = shock_h51.head(15).sort_values("delta_x")
plot_h51["sector_label"] = plot_h51["sector"].map(sector_label)
fig = px.bar(
    plot_h51,
    x="delta_x",
    y="sector_label",
    orientation="h",
    template="plotly_white",
    title="Impactos setoriais no output: choque de -1% em transporte aéreo",
    labels={"delta_x": "Δx (10^6USD)", "sector_label": "Setor"},
)
fig.update_layout(title_x=0.5, legend_title_text="", margin=dict(t=90))
fig

In [ ]:
#| label: txt-shock-h51-interpretation
#| echo: false
#| output: asis
dx_h51 = L_ext @ dy_h51
h51_spill = shock_h51[shock_h51["sector"] != "H51"].iloc[0]
h51_others = shock_h51[shock_h51["sector"] != "H51"].copy()
h51_neg_others = int((h51_others["delta_x"] < 0).sum())
h51_pos_others = int((h51_others["delta_x"] > 0).sum())
h51_zero_others = int((h51_others["delta_x"] == 0).sum())
h51_n_others = int(len(h51_others))
h51_initial = float(dy_h51.loc["H51"])
h51_indirect_own = float(dx_h51.loc["H51"] - h51_initial)
h51_total_prod = float(dx_h51.loc[S].sum())
h51_indirect_prod = float(h51_total_prod - dy_h51.loc[S].sum())

txt = f"""
Leitura económica da figura do cenário 1:

- A figura mostra **efeitos totais** ($\Delta x$), calculados por $\Delta x = (I-A_{{ext}})^{{-1}}\Delta y$, e não apenas a componente inicial.
- No setor de origem (**{sector_label('H51')}**), o efeito inicial é **Δy = {h51_initial:,.1f} milhões USD** e a componente indireta adicional é **{h51_indirect_own:,.1f} milhões USD**.
- O maior efeito indireto (excluindo o setor de origem) ocorre em **{sector_label(h51_spill['sector'])}**, com **Δx = {h51_spill['delta_x']:,.1f} milhões USD**.
- No agregado dos setores produzidos, o efeito total é **{h51_total_prod:,.1f} milhões USD**, dos quais **{h51_indirect_prod:,.1f} milhões USD** resultam de propagação indireta na rede.
- Entre os **{h51_n_others} setores restantes (excluindo o setor de origem)**, observam-se **{h51_neg_others}** impactos negativos, **{h51_pos_others}** positivos e **{h51_zero_others}** nulos. As magnitudes dos efeitos são heterogéneas entre setores.
"""
print(txt)

### Cenário 2: choque de +1% na procura final de alojamento e restauração

In [ ]:
#| label: fig-shock-i
#| fig-cap: "Cenário 2 (alojamento e restauração): impactos no output (top 15 setores)."
plot_i = shock_i.head(15).sort_values("delta_x")
plot_i["sector_label"] = plot_i["sector"].map(sector_label)
fig = px.bar(
    plot_i,
    x="delta_x",
    y="sector_label",
    orientation="h",
    template="plotly_white",
    title="Impactos setoriais no output: choque de +1% em alojamento e restauração",
    labels={"delta_x": "Δx (10^6USD)", "sector_label": "Setor"},
)
fig.update_layout(title_x=0.5, legend_title_text="", margin=dict(t=90))
fig

In [ ]:
#| label: txt-shock-i-interpretation
#| echo: false
#| output: asis
dx_i = L_ext @ dy_i
i_spill = shock_i[shock_i["sector"] != "I"].iloc[0]
i_others = shock_i[shock_i["sector"] != "I"].copy()
i_neg_others = int((i_others["delta_x"] < 0).sum())
i_pos_others = int((i_others["delta_x"] > 0).sum())
i_zero_others = int((i_others["delta_x"] == 0).sum())
i_n_others = int(len(i_others))
i_initial = float(dy_i.loc["I"])
i_indirect_own = float(dx_i.loc["I"] - i_initial)
i_total_prod = float(dx_i.loc[S].sum())
i_indirect_prod = float(i_total_prod - dy_i.loc[S].sum())

txt = f"""
Leitura económica da figura do cenário 2:

- A figura mostra **efeitos totais** ($\Delta x$), calculados por $\Delta x = (I-A_{{ext}})^{{-1}}\Delta y$, e não apenas a componente inicial.
- No setor de origem (**{sector_label('I')}**), o efeito inicial é **Δy = {i_initial:,.1f} milhões USD** e a componente indireta adicional é **{i_indirect_own:,.1f} milhões USD**.
- O maior efeito indireto (excluindo o setor de origem) surge em **{sector_label(i_spill['sector'])}**, com **Δx = {i_spill['delta_x']:,.1f} milhões USD**.
- No agregado dos setores produzidos, o efeito total é **{i_total_prod:,.1f} milhões USD**, dos quais **{i_indirect_prod:,.1f} milhões USD** resultam de propagação indireta na rede.
- Entre os **{i_n_others} setores restantes (excluindo o setor de origem)**, observam-se **{i_pos_others}** impactos positivos, **{i_neg_others}** negativos e **{i_zero_others}** nulos. As magnitudes distribuem-se de forma desigual ao longo da cadeia produtiva.
"""
print(txt)

## Add-ons macroeconómicos

Nesta secção complementamos os resultados-base com leituras mais próximas da linguagem macroeconómica usada em aulas introdutórias.

A ideia é separar três perguntas diferentes:

1. quanto output bruto total é ativado por um choque de procura final;
2. quanto desse efeito se traduz em valor acrescentado doméstico;
3. quais setores funcionam como “nós” centrais de propagação na rede.

### Multiplicador de valor acrescentado (VA) vs. multiplicador de output

No modelo de Leontief, o primeiro passo é sempre obter o efeito total em produção:

$$
\Delta x = (I-A)^{-1}\Delta y.
$$

Este $\Delta x$ mede **produção bruta** (gross output). Em contabilidade nacional, produção bruta não é o mesmo que contribuição para rendimento, porque inclui consumo intermédio entre setores.

Para aproximar a componente de rendimento doméstico, usamos os coeficientes de VA por unidade de output:

$$
\Delta VA \approx \sum_{i \in S} v_i \Delta x_i,
\qquad
v_i = \frac{VA_i}{x_i},
$$

e definimos dois multiplicadores para o choque no setor $j$:

$$
m_{X,j} = \frac{\sum_{i \in S}\Delta x_i}{\Delta y_j},
\qquad
m_{VA,j} = \frac{\sum_{i \in S} v_i\Delta x_i}{\Delta y_j}.
$$

Leitura pedagógica:

- $m_X$ responde “quanto output total a economia produz por 1 unidade de choque final”;
- $m_{VA}$ responde “quanto valor acrescentado doméstico é gerado por 1 unidade de choque final”.

In [ ]:
#| label: tbl-va-multipliers
#| tbl-cap: "Comparação entre multiplicadores de output e de valor acrescentado (VA)."
va_mult_tbl = scenario_summary.copy()
va_mult_tbl["Setor do choque"] = va_mult_tbl["setor_codigo"].map(lambda c: f"{sector_label(c)} ({c})")
va_mult_tbl = va_mult_tbl[
    [
        "Setor do choque",
        "choque_pct",
        "delta_y",
        "delta_x_total",
        "delta_va",
        "output_multiplier",
        "va_multiplier",
    ]
].rename(
    columns={
        "choque_pct": "Choque (%)",
        "delta_y": "Δy (10^6USD)",
        "delta_x_total": "ΣΔx (10^6USD)",
        "delta_va": "ΔVA (10^6USD)",
        "output_multiplier": "Multiplicador de output",
        "va_multiplier": "Multiplicador de VA",
    }
)

(
    va_mult_tbl.style.hide(axis="index")
    .format(
        {
            "Choque (%)": "{:.2f}",
            "Δy (10^6USD)": "{:,.2f}",
            "ΣΔx (10^6USD)": "{:,.2f}",
            "ΔVA (10^6USD)": "{:,.2f}",
            "Multiplicador de output": "{:.4f}",
            "Multiplicador de VA": "{:.4f}",
        }
    )
    .set_properties(subset=["Setor do choque"], **{"text-align": "left"})
    .set_properties(
        subset=[
            "Choque (%)",
            "Δy (10^6USD)",
            "ΣΔx (10^6USD)",
            "ΔVA (10^6USD)",
            "Multiplicador de output",
            "Multiplicador de VA",
        ],
        **{"text-align": "right"},
    )
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {
                "selector": "th.col_heading.col1, th.col_heading.col2, th.col_heading.col3, th.col_heading.col4, th.col_heading.col5, th.col_heading.col6",
                "props": [("text-align", "right")],
            },
        ],
        overwrite=False,
    )
)

In [ ]:
#| label: txt-va-multiplier-interpretation
#| echo: false
#| output: asis
h51_row = scenario_lookup.loc["H51"]
i_row = scenario_lookup.loc["I"]
h51_gap = float(h51_row["output_multiplier"] - h51_row["va_multiplier"])
i_gap = float(i_row["output_multiplier"] - i_row["va_multiplier"])

txt = f"""
Leitura económica da tabela acima:

- Em **{sector_label('H51')}**, $m_X = {h51_row['output_multiplier']:.3f}$ e $m_{{VA}} = {h51_row['va_multiplier']:.3f}$; a diferença é **{h51_gap:.3f}**.
- Em **{sector_label('I')}**, $m_X = {i_row['output_multiplier']:.3f}$ e $m_{{VA}} = {i_row['va_multiplier']:.3f}$; a diferença é **{i_gap:.3f}**.
- O hiato $m_X - m_{{VA}}$ capta a parcela do efeito total que permanece em fluxos intermédios entre setores (não em VA).
- Em aula, esta distinção ajuda a evitar a leitura incorreta de multiplicador de produção bruta como “multiplicador de PIB”.
"""
print(txt)

### Decomposição: VA doméstico, cunha fiscal (`TXS`) e fuga importada (`IMP_OTHER`)

Nesta decomposição, os fluxos monetários ($\Delta VA$, $\Delta x_{TXS}$ e $\Delta x_{IMP\_OTHER}$) estão em **milhões de USD**.  
As razões por unidade de choque ($\Delta x_{TXS}/\Delta y$ e $\Delta x_{IMP\_OTHER}/\Delta y$) são adimensionais e ajudam a comparar setores com escalas diferentes.

In [ ]:
#| label: tbl-decomposition-wedges
#| tbl-cap: "Decomposição dos efeitos dos choques: VA, TXS e IMP_OTHER."
plus_shocks = sandbox_summary[sandbox_summary["shock_pct"] == 1.0].copy()
plus_shocks_nonbase = plus_shocks[~plus_shocks["shock_sector"].isin(["H51", "I"])].copy()
if plus_shocks_nonbase.empty:
    plus_shocks_nonbase = plus_shocks.copy()

txs_pick = plus_shocks_nonbase.sort_values("tax_wedge_ratio", ascending=False).iloc[0]
imp_sorted = plus_shocks_nonbase.sort_values("import_leakage_ratio", ascending=False)
imp_candidates = imp_sorted[imp_sorted["shock_sector"] != txs_pick["shock_sector"]]
imp_pick = imp_candidates.iloc[0] if not imp_candidates.empty else imp_sorted.iloc[0]

selected_specs = [
    {"code": "H51", "pct": -1.0, "criterion": "Cenário base"},
    {"code": "I", "pct": 1.0, "criterion": "Cenário base"},
    {"code": str(txs_pick["shock_sector"]), "pct": 1.0, "criterion": "Alto Δx_TXS / Δy (+1%)"},
]

if imp_pick["shock_sector"] != txs_pick["shock_sector"]:
    selected_specs.append(
        {"code": str(imp_pick["shock_sector"]), "pct": 1.0, "criterion": "Alto Δx_IMP_OTHER / Δy (+1%)"}
    )
else:
    selected_specs[2]["criterion"] = "Alto Δx_TXS / Δy e Δx_IMP_OTHER / Δy (+1%)"

decomp_rows = []
for spec in selected_specs:
    code = spec["code"]
    pct = spec["pct"]

    from_main = scenario_summary[
        (scenario_summary["setor_codigo"] == code) & (np.isclose(scenario_summary["choque_pct"], pct))
    ]

    if not from_main.empty:
        base = from_main.iloc[0]
        row = {
            "setor_codigo": code,
            "choque_pct": float(base["choque_pct"]),
            "delta_va": float(base["delta_va"]),
            "delta_txs": float(base["delta_txs"]),
            "delta_imp_other": float(base["delta_imp_other"]),
            "va_share_in_delta_x": float(base["va_share_in_delta_x"]),
            "txs_per_delta_y": float(base["txs_per_delta_y"]),
            "imp_other_per_delta_y": float(base["imp_other_per_delta_y"]),
        }
    else:
        from_sandbox = sandbox_summary[
            (sandbox_summary["shock_sector"] == code) & (np.isclose(sandbox_summary["shock_pct"], pct))
        ]
        if from_sandbox.empty:
            continue
        base = from_sandbox.iloc[0]
        delta_x_total = float(base["delta_x_total"])
        row = {
            "setor_codigo": code,
            "choque_pct": float(base["shock_pct"]),
            "delta_va": float(base["delta_va"]),
            "delta_txs": float(base["delta_txs"]),
            "delta_imp_other": float(base["delta_imp_other"]),
            "va_share_in_delta_x": np.nan if np.isclose(delta_x_total, 0.0) else float(base["delta_va"]) / delta_x_total,
            "txs_per_delta_y": float(base["tax_wedge_ratio"]),
            "imp_other_per_delta_y": float(base["import_leakage_ratio"]),
        }

    row["criterion"] = spec["criterion"]
    decomp_rows.append(row)

decomp_calc = pd.DataFrame(decomp_rows)
decomp_tbl = decomp_calc.copy()
decomp_tbl["Setor do choque"] = decomp_tbl["setor_codigo"].map(lambda c: f"{sector_label(c)} ({c})")
decomp_tbl["Critério"] = decomp_tbl["criterion"]
decomp_tbl["Choque (%)"] = decomp_tbl["choque_pct"]
decomp_tbl = decomp_tbl[
    [
        "Setor do choque",
        "Critério",
        "Choque (%)",
        "delta_va",
        "delta_txs",
        "delta_imp_other",
        "va_share_in_delta_x",
        "txs_per_delta_y",
        "imp_other_per_delta_y",
    ]
].rename(
    columns={
        "delta_va": "ΔVA (10^6USD)",
        "delta_txs": "Δx_TXS (10^6USD)",
        "delta_imp_other": "Δx_IMP_OTHER (10^6USD)",
        "va_share_in_delta_x": "ΔVA / ΣΔx",
        "txs_per_delta_y": "Δx_TXS / Δy",
        "imp_other_per_delta_y": "Δx_IMP_OTHER / Δy",
    }
)

(
    decomp_tbl.style.hide(axis="index")
    .format(
        {
            "Choque (%)": "{:.2f}",
            "ΔVA (10^6USD)": "{:,.2f}",
            "Δx_TXS (10^6USD)": "{:,.2f}",
            "Δx_IMP_OTHER (10^6USD)": "{:,.2f}",
            "ΔVA / ΣΔx": "{:.4f}",
            "Δx_TXS / Δy": "{:.4f}",
            "Δx_IMP_OTHER / Δy": "{:.4f}",
        }
    )
    .set_properties(subset=["Setor do choque", "Critério"], **{"text-align": "left"})
    .set_properties(
        subset=[
            "Choque (%)",
            "ΔVA (10^6USD)",
            "Δx_TXS (10^6USD)",
            "Δx_IMP_OTHER (10^6USD)",
            "ΔVA / ΣΔx",
            "Δx_TXS / Δy",
            "Δx_IMP_OTHER / Δy",
        ],
        **{"text-align": "right"},
    )
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0, th.col_heading.col1", "props": [("text-align", "left")]},
            {
                "selector": "th.col_heading.col2, th.col_heading.col3, th.col_heading.col4, th.col_heading.col5, th.col_heading.col6, th.col_heading.col7, th.col_heading.col8",
                "props": [("text-align", "right")],
            },
        ],
        overwrite=False,
    )
)

In [ ]:
#| label: txt-decomposition-interpretation
#| echo: false
#| output: asis
h51_row = decomp_calc.loc[decomp_calc["setor_codigo"] == "H51"].iloc[0]
i_row = decomp_calc.loc[decomp_calc["setor_codigo"] == "I"].iloc[0]
txs_focus = decomp_calc[decomp_calc["criterion"].str.contains("TXS", regex=False)].iloc[0]
imp_focus_candidates = decomp_calc[decomp_calc["criterion"].str.contains("IMP_OTHER", regex=False)]
imp_focus = imp_focus_candidates.iloc[0] if not imp_focus_candidates.empty else txs_focus

txt = f"""
Leitura económica da tabela acima:

- As duas primeiras linhas mantêm os cenários-base (choques em **{sector_label('H51')}** e **{sector_label('I')}**), para facilitar comparação com as secções anteriores.
- A linha com critério de alta razão **Δx_TXS/Δy** destaca **{sector_label(txs_focus['setor_codigo'])}**, com valor **{txs_focus['txs_per_delta_y']:.3f}** para um choque de +1%.
- A linha com critério de alta razão **Δx_IMP_OTHER/Δy** destaca **{sector_label(imp_focus['setor_codigo'])}**, com valor **{imp_focus['imp_other_per_delta_y']:.3f}** para um choque de +1%.
- Em termos económicos: **Δx_TXS/Δy** alto indica maior transmissão para a cunha fiscal sobre produtos; **Δx_IMP_OTHER/Δy** alto indica maior sensibilidade a conteúdo importado “extra-setorial”.
- Nos cenários-base, **Δx_IMP_OTHER/Δy = {h51_row['imp_other_per_delta_y']:.3f}** em {sector_label('H51')} e **{i_row['imp_other_per_delta_y']:.3f}** em {sector_label('I')}.
"""
print(txt)

### Setores-chave: encadeamentos backward e forward

Definimos:

$$
\text{Backward}_j = \sum_i l_{ij},
\qquad
\text{Forward}_i = \sum_j l_{ij},
$$

onde $l_{ij}$ são os elementos da matriz inversa de Leontief sobre setores produzidos.

Interpretação prática:

- **Backward linkage** de $j$: 
  $$
  \text{Backward}_j = \sum_i l_{ij} = \sum_i \frac{\partial x_i}{\partial y_j}.
  $$
  É a variação do **output total da economia** $\sum_i \Delta x_i$ quando aplicamos um choque unitário em procura final no setor $j$ (isto é, $\Delta y_j=1$ e $\Delta y_{k\neq j}=0$).
- **Forward linkage** de $i$:
  $$
  \text{Forward}_i = \sum_j l_{ij} = \sum_j \frac{\partial x_i}{\partial y_j}.
  $$
  É a variação do **output do setor $i$** quando todas as componentes da procura final recebem choques unitários e somamos esses efeitos (equivalentemente, quando $\Delta y=\mathbf{1}$, então $\Delta x_i=\text{Forward}_i$).

Em linguagem de política:

- setores com backward alto tendem a ser candidatos a maior “efeito multiplicador de procura”;
- setores com forward alto tendem a ser pontos críticos para gestão de choques de custos/oferta.

In [ ]:
#| label: tbl-backward-top10
#| tbl-cap: "Top 10 setores por backward linkage (soma por coluna de L)."
(
    backward_top10.style.hide(axis="index")
    .format({"backward_linkage": "{:.4f}"})
    .set_properties(subset=["Setor"], **{"text-align": "left"})
    .set_properties(subset=["backward_linkage"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {"selector": "th.col_heading.col1", "props": [("text-align", "right")]},
        ],
        overwrite=False,
    )
)

In [ ]:
#| label: tbl-forward-top10
#| tbl-cap: "Top 10 setores por forward linkage (soma por linha de L)."
(
    forward_top10.style.hide(axis="index")
    .format({"forward_linkage": "{:.4f}"})
    .set_properties(subset=["Setor"], **{"text-align": "left"})
    .set_properties(subset=["forward_linkage"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {"selector": "th.col_heading.col1", "props": [("text-align", "right")]},
        ],
        overwrite=False,
    )
)

In [ ]:
#| label: txt-linkages-interpretation
#| echo: false
#| output: asis
top_back = backward_top10.iloc[0]
second_back = backward_top10.iloc[1]
top_forw = forward_top10.iloc[0]
second_forw = forward_top10.iloc[1]

txt = f"""
Leitura económica das tabelas acima:

- O maior **backward linkage** é de **{top_back['Setor']}** (**{top_back['backward_linkage']:.3f}**), seguido de **{second_back['Setor']}** (**{second_back['backward_linkage']:.3f}**).
- O maior **forward linkage** é de **{top_forw['Setor']}** (**{top_forw['forward_linkage']:.3f}**), seguido de **{second_forw['Setor']}** (**{second_forw['forward_linkage']:.3f}**).
- Para leitura didática: o ranking backward ajuda a discutir estímulos de procura; o ranking forward ajuda a discutir vulnerabilidade a estrangulamentos na oferta.
"""
print(txt)

### Sandbox interativo (setor, tamanho e unidade do choque)

Use o sandbox abaixo para simular choques de procura final com três escolhas:

1. setor do choque (campo pesquisável);
2. tamanho do choque;
3. unidade do choque (`% de y` do setor ou valor absoluto em `Milhões USD`).

Depois de selecionar os inputs, o painel devolve os impactos no output setorial (top 15) e um resumo macro dos efeitos totais.

In [ ]:
#| label: fig-sandbox-dropdown
#| echo: false
#| output: asis
sandbox_payload = {
    "sectors": [
        {
            "code": c,
            "label": sector_label(c),
            "display": f"{sector_label(c)} ({c})",
            "y": float(y_ext.loc[c]),
            "x": float(x_ext.loc[c]),
        }
        for c in S
    ],
    "sector_order": list(S),
    "sector_display_order": [f"{sector_label(c)} ({c})" for c in S],
    "responses": {c: [float(v) for v in L_ext.loc[S, c].values] for c in S},
    "v_share": [float(v_share.loc[c]) for c in S],
    "l_txs": {c: float(L_ext.loc["TXS", c]) for c in S},
    "l_imp_other": {c: float(L_ext.loc["IMP_OTHER", c]) for c in S},
}
sandbox_json = json.dumps(sandbox_payload, ensure_ascii=False)

sandbox_html = """
<div id="io-sandbox-app"></div>
<script>
(function () {
  const payload = """ + sandbox_json + """;
  const app = document.getElementById("io-sandbox-app");
  if (!app) return;

  app.innerHTML = `
    <div style="display:grid;grid-template-columns:2fr 1fr 1fr auto;gap:0.7rem;align-items:end;margin:0.5rem 0 0.8rem 0;">
      <div>
        <label for="io-sector-input"><strong>Setor do choque</strong></label>
        <input id="io-sector-input" list="io-sector-list" placeholder="Ex.: Transporte aéreo (H51)" style="width:100%;padding:0.4rem;" />
        <datalist id="io-sector-list"></datalist>
      </div>
      <div>
        <label for="io-shock-size"><strong>Tamanho do choque</strong></label>
        <input id="io-shock-size" type="number" step="0.1" value="1" style="width:100%;padding:0.4rem;" />
      </div>
      <div>
        <label for="io-shock-unit"><strong>Unidade</strong></label>
        <select id="io-shock-unit" style="width:100%;padding:0.4rem;">
          <option value="pct">% de y (procura final do setor)</option>
          <option value="abs">Milhões USD (Δy absoluto)</option>
        </select>
      </div>
      <div>
        <button id="io-run-btn" style="padding:0.45rem 0.9rem;">Atualizar</button>
      </div>
    </div>
    <div id="io-sandbox-alert" style="display:none;color:#b00020;font-size:0.95rem;margin-bottom:0.5rem;"></div>
    <div id="io-sandbox-summary" style="margin:0.4rem 0 0.9rem 0;"></div>
    <div id="io-sandbox-chart" style="min-height:520px;"></div>
  `;

  if (typeof Plotly === "undefined") {
    const alertBox = document.getElementById("io-sandbox-alert");
    alertBox.style.display = "block";
    alertBox.textContent = "Plotly não está disponível nesta página.";
    return;
  }

  const sectors = payload.sectors;
  const byCode = Object.fromEntries(sectors.map(s => [s.code, s]));
  const byDisplay = Object.fromEntries(sectors.map(s => [s.display.toLowerCase(), s.code]));

  const sectorInput = document.getElementById("io-sector-input");
  const sectorList = document.getElementById("io-sector-list");
  const shockSizeInput = document.getElementById("io-shock-size");
  const shockUnitSelect = document.getElementById("io-shock-unit");
  const runBtn = document.getElementById("io-run-btn");
  const alertBox = document.getElementById("io-sandbox-alert");
  const summaryBox = document.getElementById("io-sandbox-summary");
  const chartDiv = document.getElementById("io-sandbox-chart");

  sectors.forEach(s => {
    const option = document.createElement("option");
    option.value = s.display;
    sectorList.appendChild(option);
  });

  const defaultCode = byCode["H51"] ? "H51" : sectors[0].code;
  sectorInput.value = byCode[defaultCode].display;

  const fmt2 = new Intl.NumberFormat("pt-PT", { minimumFractionDigits: 2, maximumFractionDigits: 2 });
  const fmt3 = new Intl.NumberFormat("pt-PT", { minimumFractionDigits: 3, maximumFractionDigits: 3 });

  function resolveSector(rawValue) {
    const raw = (rawValue || "").trim();
    if (!raw) return null;
    if (byCode[raw]) return raw;
    const exact = byDisplay[raw.toLowerCase()];
    if (exact) return exact;
    const match = raw.match(/\\(([A-Za-z0-9_]+)\\)\\s*$/);
    if (match && byCode[match[1]]) return match[1];
    const approx = sectors.find(s => s.label.toLowerCase() === raw.toLowerCase());
    return approx ? approx.code : null;
  }

  function renderSummary(code, unit, shockSize, deltaY, deltaX, deltaVA, deltaTXS, deltaImpOther) {
    const totalDX = deltaX.reduce((acc, v) => acc + v, 0);
    const outputMult = deltaY === 0 ? NaN : totalDX / deltaY;
    const vaMult = deltaY === 0 ? NaN : deltaVA / deltaY;
    const impLeak = deltaY === 0 ? NaN : deltaImpOther / deltaY;
    const taxWedge = deltaY === 0 ? NaN : deltaTXS / deltaY;
    const unitTxt = unit === "pct" ? `${fmt2.format(shockSize)}% de y` : `${fmt2.format(shockSize)} (Milhões USD)`;

    summaryBox.innerHTML = `
      <table class="table table-sm table-striped" style="margin-bottom:0;">
        <tbody>
          <tr><th style="text-align:left;">Setor do choque</th><td style="text-align:right;">${byCode[code].display}</td></tr>
          <tr><th style="text-align:left;">Input do choque</th><td style="text-align:right;">${unitTxt}</td></tr>
          <tr><th style="text-align:left;">Δy aplicado (Milhões USD)</th><td style="text-align:right;">${fmt2.format(deltaY)}</td></tr>
          <tr><th style="text-align:left;">ΣΔx (Milhões USD)</th><td style="text-align:right;">${fmt2.format(totalDX)}</td></tr>
          <tr><th style="text-align:left;">ΔVA (Milhões USD)</th><td style="text-align:right;">${fmt2.format(deltaVA)}</td></tr>
          <tr><th style="text-align:left;">Δx_TXS (Milhões USD)</th><td style="text-align:right;">${fmt2.format(deltaTXS)}</td></tr>
          <tr><th style="text-align:left;">Δx_IMP_OTHER (Milhões USD)</th><td style="text-align:right;">${fmt2.format(deltaImpOther)}</td></tr>
          <tr><th style="text-align:left;">Multiplicador de output</th><td style="text-align:right;">${Number.isFinite(outputMult) ? fmt3.format(outputMult) : "n.d."}</td></tr>
          <tr><th style="text-align:left;">Multiplicador de VA</th><td style="text-align:right;">${Number.isFinite(vaMult) ? fmt3.format(vaMult) : "n.d."}</td></tr>
          <tr><th style="text-align:left;">Δx_TXS / Δy</th><td style="text-align:right;">${Number.isFinite(taxWedge) ? fmt3.format(taxWedge) : "n.d."}</td></tr>
          <tr><th style="text-align:left;">Δx_IMP_OTHER / Δy</th><td style="text-align:right;">${Number.isFinite(impLeak) ? fmt3.format(impLeak) : "n.d."}</td></tr>
        </tbody>
      </table>
    `;
  }

  function updateSandbox() {
    alertBox.style.display = "none";
    alertBox.textContent = "";

    const code = resolveSector(sectorInput.value);
    const shockSize = Number(shockSizeInput.value);
    const unit = shockUnitSelect.value;

    if (!code) {
      alertBox.style.display = "block";
      alertBox.textContent = "Setor inválido. Escolha um setor da lista pesquisável.";
      return;
    }
    if (!Number.isFinite(shockSize)) {
      alertBox.style.display = "block";
      alertBox.textContent = "Tamanho do choque inválido. Introduza um valor numérico.";
      return;
    }

    const sectorInfo = byCode[code];
    const deltaY = unit === "pct" ? (shockSize / 100.0) * sectorInfo.y : shockSize;

    const responseCol = payload.responses[code];
    const deltaX = responseCol.map(v => v * deltaY);
    const deltaVA = deltaX.reduce((acc, v, idx) => acc + payload.v_share[idx] * v, 0.0);
    const deltaTXS = payload.l_txs[code] * deltaY;
    const deltaImpOther = payload.l_imp_other[code] * deltaY;

    const rows = payload.sector_order.map((secCode, idx) => ({
      sector: secCode,
      label: payload.sector_display_order[idx],
      delta_x: deltaX[idx],
      abs_delta_x: Math.abs(deltaX[idx]),
    }));

    rows.sort((a, b) => b.abs_delta_x - a.abs_delta_x);
    const top15 = rows.slice(0, 15).sort((a, b) => a.delta_x - b.delta_x);

    const shockTxt = unit === "pct"
      ? `${fmt2.format(shockSize)}% de y`
      : `${fmt2.format(shockSize)} (Milhões USD) em Δy`;

    const trace = {
      type: "bar",
      orientation: "h",
      x: top15.map(r => r.delta_x),
      y: top15.map(r => r.label),
      marker: { color: "#2f6f87" },
      hovertemplate: "%{y}<br>Δx=%{x:.2f} (Milhões USD)<extra></extra>",
    };

    const layout = {
      template: "plotly_white",
      title: `Sandbox IO: choque em ${sectorInfo.display} | ${shockTxt}`,
      title_x: 0.5,
      margin: { t: 90, r: 20, b: 60, l: 220 },
      xaxis: { title: "Δx (Milhões USD)" },
      yaxis: { title: "Setor" },
    };

    Plotly.react(chartDiv, [trace], layout, { responsive: true, displayModeBar: false });
    renderSummary(code, unit, shockSize, deltaY, deltaX, deltaVA, deltaTXS, deltaImpOther);
  }

  runBtn.addEventListener("click", updateSandbox);
  sectorInput.addEventListener("change", updateSandbox);
  shockSizeInput.addEventListener("change", updateSandbox);
  shockUnitSelect.addEventListener("change", updateSandbox);
  shockSizeInput.addEventListener("keydown", function (ev) {
    if (ev.key === "Enter") updateSandbox();
  });

  updateSandbox();
})();
</script>
"""

print(sandbox_html)

In [ ]:
#| label: txt-sandbox-interpretation
#| echo: false
#| output: asis
txt = f"""
Leitura económica do sandbox:

- O cálculo é sempre **$\Delta x = (I-A)^{{-1}}\Delta y$**: os resultados incluem efeito inicial e propagação em rede.
- Se escolher a unidade `% de y`, o sandbox converte automaticamente o valor em choque absoluto de procura final para o setor selecionado.
- Se escolher `Milhões USD`, o choque entra diretamente como $\Delta y$ absoluto.
- O quadro-resumo mostra, para o mesmo choque, produção total, valor acrescentado e decomposição para `TXS` e `IMP_OTHER`, o que ajuda a comparar setores com maior transmissão fiscal/importada.
"""
print(txt)

### Mini-caso de política: aumento do custo importado (`IMP_OTHER`)

Consideramos um choque de preços exógenos nas cunhas:

$$
\Delta p_W = (\Delta p_{\text{TXS}}, \Delta p_{\text{IMP\_OTHER}})^\top = (0,\ 0.10)^\top.
$$

Com coeficientes fixos, o efeito em custos unitários dos setores produzidos é:

$$
\Delta p_S = (I - A_{SS}^{\top})^{-1}A_{WS}^{\top}\Delta p_W.
$$

In [ ]:
#| label: tbl-import-cost-mini-case
#| tbl-cap: "Mini-caso: principais setores afetados por um aumento de 10% em IMP_OTHER."
mini_case_rank = import_cost_case.copy()
mini_case_rank["Setor"] = mini_case_rank["sector"].map(lambda c: f"{sector_label(c)} ({c})")
mini_case_rank = mini_case_rank.sort_values("delta_p_hat_pct_base", ascending=False).head(15)
mini_case_tbl = mini_case_rank[["Setor", "delta_p_hat_pct_base"]].rename(
    columns={"delta_p_hat_pct_base": "Δp_hat / p_hat_base (%)"}
)

(
    mini_case_tbl.style.hide(axis="index")
    .format({"Δp_hat / p_hat_base (%)": "{:.2f}"})
    .set_properties(subset=["Setor"], **{"text-align": "left"})
    .set_properties(subset=["Δp_hat / p_hat_base (%)"], **{"text-align": "right"})
    .set_table_styles(
        [
            {"selector": "th.col_heading.col0", "props": [("text-align", "left")]},
            {"selector": "th.col_heading.col1", "props": [("text-align", "right")]},
        ],
        overwrite=False,
    )
)

In [ ]:
#| label: txt-import-cost-interpretation
#| echo: false
#| output: asis
top_imp_case = mini_case_rank.iloc[0]

txt = f"""
Leitura económica do mini-caso:

- Um aumento exógeno do custo de `IMP_OTHER` propaga-se para os custos setoriais via matriz transposta de coeficientes técnicos.
- A tabela ordena os setores do maior para o menor impacto relativo, medido por **Δp_hat / p_hat_base (%)**.
- O setor mais sensível neste exercício é **{top_imp_case['Setor']}**, com variação relativa de **{top_imp_case['delta_p_hat_pct_base']:.2f}%** face ao seu `p_hat` de base.
"""
print(txt)

## Limitações

- A análise é estática e de curto prazo: os coeficientes técnicos são fixos.
- Não há substituição entre inputs, restrições de capacidade nem respostas comportamentais.
- Os choques são exercícios contábeis de propagação na rede produtiva, não previsões estruturais completas.

## Conclusão

A matriz técnica e a inversa de Leontief fornecem uma forma transparente de ligar choques de procura final a efeitos totais na produção setorial. Para ensino introdutório, os exercícios em transporte aéreo e alojamento/restauração ajudam a visualizar encadeamentos produtivos e a diferença entre efeito inicial e propagação indireta.

In [ ]:
#| label: txt-conclusion-numbers
#| echo: false
#| output: asis
h51_mult = float(multiplier_summary.loc[multiplier_summary["setor_codigo"] == "H51", "multiplicador_output_total"].iloc[0])
i_mult = float(multiplier_summary.loc[multiplier_summary["setor_codigo"] == "I", "multiplicador_output_total"].iloc[0])

h51_dy = float(dy_h51.loc["H51"])
i_dy = float(dy_i.loc["I"])
h51_dx_total = float((L_ext @ dy_h51).loc[S].sum())
i_dx_total = float((L_ext @ dy_i).loc[S].sum())

h51_spill = shock_h51[shock_h51["sector"] != "H51"].iloc[0]
i_spill = shock_i[shock_i["sector"] != "I"].iloc[0]

txt = f"""
Leitura numérica dos resultados principais:

- O choque de **-1%** na procura final de **{sector_label('H51')}** corresponde a **Δy = {h51_dy:,.1f} milhões USD** e implica uma variação total no output de **{h51_dx_total:,.1f} milhões USD** (multiplicador agregado **{h51_mult:.3f}**).
- O choque de **+1%** na procura final de **{sector_label('I')}** corresponde a **Δy = {i_dy:,.1f} milhões USD** e implica uma variação total no output de **{i_dx_total:,.1f} milhões USD** (multiplicador agregado **{i_mult:.3f}**).
- No cenário de transporte aéreo, o maior efeito de propagação (excluindo o próprio setor) surge em **{sector_label(h51_spill['sector'])}**, com **Δx = {h51_spill['delta_x']:,.1f} milhões USD**.
- No cenário de alojamento/restauração, o maior efeito de propagação (excluindo o próprio setor) surge em **{sector_label(i_spill['sector'])}**, com **Δx = {i_spill['delta_x']:,.1f} milhões USD**.

Estes números reforçam a intuição económica: mesmo choques concentrados num único setor geram efeitos distribuídos por vários setores devido às ligações de input-output.
"""
print(txt)